In [ ]:
import os
import sys
import glob
import pandas as pd

#avoid import errors
parent_dir=os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

from scripts.CDS_machine import CDS_2_gffcomp
from scripts.statsCompression import gffstats_2data
import scripts.get_busco_db as bob
import scripts.get_genetic_code as ggc


In [ ]:
raw_data="../../data/species"
merged_data="../results/merged"
query_email=""
busco_database="../../data/busco_downloads/"

!ls $merged_data

Babesia_duncani_323732
Chaetoceros_neogracilis_240364
Chlamydomonas_reinhardtii_3055
Cladocopium_goreaui_2562237
Conticribra_weissflogii_1577725
Cryptosporidium_parvum_Iowa_II_353152
Cyanidiococcus_yangmingshanensis_2690220
Cyanidioschyzon_merolae_strain_10d_280699
Cyanidioschyzon_merolae_strain_10D_280699
Cyanidium_caldarium_2771
Cylindrotheca_closterium_2856
Dunaliella_salina_3046
Durusdinium_trenchii_1381693
Eimeria_necatrix_51315
Emiliania_huxleyi_CCMP1516_280463
Entamoeba_histolytica_HM-1:IMSS_294381
Galdieria_yellowstonensis_3028027
Giardia_duodenalis_5741
Giardia_muris_5742
Haematococcus_lacustris_44745
Hamiltosporidium_tvaerminnensis_1176355
Leishmania_donovani_5661
Leishmania_infantum_JPCM5_435258
Micromonas_pusilla_CCMP1545_564608
Naegleria_gruberi_5762
Paramecium_bursaria_74790
Paramecium_tetraurelia_5888
Plasmodium_berghei_ANKA_5823
Plasmodium_falciparum_3D7_36329
Plasmodium_vivax_5855
Plasmodium_yoelii_5861
Symbiodinium_natans_878477
Symbiodinium_necroappetens_1628268
Tetr

# Run GeneID

In [ ]:
#geneid_command="time geneid -3P {parameter_file} {assembly} > {annotation.gff3}"

tr_sp=!ls $merged_data
print(len(tr_sp))

for src in ["own", "git"]:
    #checker(if things in file)
    fileWritten=False
    #generate 1 file per source
    filepath=f"../job_commands/mrg_SARpred_{src}.txt"
    with open(filepath, "w") as out:
        #create predictions folder command
        pred_folder="../results/refined_pred"
        print(f"mkdir -p {pred_folder}", file=out)
        for sp in tr_sp:
            sp=sp.replace("_own.gff", "")
            
            #reference and model name are adaptable to the source
            ref=!realpath ../training_data/species/$sp/CLEAN_*.fna
            model=f"{pred_folder}/{sp}_{src}.gff3"
            
            #parameter location changes depending on source
            param=!realpath ../results/refined_trainedParams/$sp*.param
            param=param[0]
            #use github parameters
            if src=="git":
                filename_git=f"../training_data/geneid-parameter-files/parameter_files/{sp}*.param"

                try: #if no github parameters exist, dont runn geneID for git parameters

                    found_files=glob.glob(filename_git) #supress error output
                    #set up error for species with no github parameters
                    if not found_files: #found files is exactly filename_git if nothing is found
                        raise FileNotFoundError(f"There is no github parameters for this species")
                    
                except Exception as e:
                    print(f"{sp} presents error: {e}")
                    continue

                param=os.path.realpath(found_files[0])
            fileWritten=True #if any species has git parameters the _git files will be written

            #geneid param_own/param_git fasta_assembly > gff_model
            command=f"time geneid -3P {param} {ref[0]} > {model}"

            #link inside the specie folder just in case
            logsDir_cmd=f"mkdir -p ../results/specie_logs/{sp}/merged" #if running from exsting parameters create logs folder
            lnPred_cmd=f"ln -vf {model} ../results/specie_logs/{sp}/merged"

            print(command)
            print(command, file=out)
            print(logsDir_cmd, file=out)
            print(lnPred_cmd, file=out)
            #in the end of the execution delete empty files
            print(f"find {pred_folder} -size 0 -print -delete", file=out)

    if not fileWritten: #if the file is not written, delete it
        os.remove(filepath)

#Execute commands to predict CDS

FileNotFoundError: [Errno 2] No such file or directory: '../job_commands/SARpred_own.txt'

In [1]:
!ls $pred_folder

1-GeneIDTrain.ipynb	  3-GeneIDPredict.ipynb
2-GeneIDGitCompare.ipynb  __pycache__


# Prediction analysis

## Reference annotation(CDS) and Prediction comparison

In [ ]:

tr_sp=!ls $merged_data
pred_items=os.listdir("../results/refined_pred")
pred_items=[tri.replace("_own.gff3", "_own.gff") for tri in pred_items] #merged data is in _own.gff formats
tr_sp=[species.replace("_own.gff", "") for species in tr_sp if species in pred_items]
print(len(tr_sp))

for src in ["own", "git"]:
    #checker(if things in file)
    fileWritten=False
    filepath=f"../job_commands/refined_gffcomp_{src}.txt"
    with open(filepath, "w") as f:
        #enable extglob
        print("shopt -s extglob", file=f)
        #create folder to store comparison information
        resloc=f"../results/summary/refined/gffcmp"
        print(f"mkdir -p {resloc}", file=f)
        for sp in tr_sp: 
            #reference annotation
            RefAnn=!realpath ../training_data/species/$sp/CDS_*.gff
            RefAnn=RefAnn[0]

            #annotation prediction by source (own or git)
            pred=!realpath ../results/refined_pred/$sp*_$src*.gff
            pred=pred[0]

            #detect if github source, if there is, use github predictions if they exist
            if src=="git":
                filename_git=f"../results/pred/{sp}_{src}*_own.gff3"

                try: #if no github predictions exist, dont runn gffcompare for git preds
                    found_files=glob.glob(filename_git) #supress error output
                    #set up error for species with no github preds
                    if not found_files: #found files is exactly filename_git if nothing is found
                        raise FileNotFoundError(f"There is no github predictions for this species")
                    
                except Exception as e:
                    print(f"{sp} presents error: {e}")
                    continue

                pred=os.path.realpath(found_files[0])
            fileWritten=True #if any species has git predictionss the _git files will be written
            
            #adapt cds to be able to be compared
            adapted_RefAnn=f"../results/specie_logs/{sp}/merged/CDSgff_{sp}.gff"
            
            CDS_2_gffcomp(RefAnn, adapted_RefAnn)

            #comparison command
            comparing_cmd=f"gffcompare -r {adapted_RefAnn} {pred} -o {resloc}/{sp}-cmp"
            
            #move all except stats, which is linked
            moveComparisons_cmd=f"mv {resloc}/!(*.stats*) ../results/specie_logs/{sp}/merged"
            lnStat_cmd=f"ln -vf {resloc}/{sp}-cmp.stats ../results/specie_logs/{sp}/merged"
            
            print(comparing_cmd, file=f)
            print(comparing_cmd)
            print(moveComparisons_cmd, file=f)
            print(lnStat_cmd, file=f)
            print(f'echo "done_{sp}"', file=f)
            
    if not fileWritten: #if the file is not written, delete it
        os.remove(filepath)

In [ ]:
#Obtain all gffcmp results as 1 unified table
tr_sp=!ls $raw_data
pred_items=os.listdir("../results/refined_pred")
pred_items=[tri.replace("_own.gff3", "_own.gff") for tri in pred_items] #merged data is in _own.gff formats
tr_sp=[species.replace("_own.gff", "") for species in tr_sp if species in pred_items]
print(len(tr_sp))

statsData=[]
for sp in tr_sp:
    try:
        print(f"Species is {sp}")
        statsfile=f"../results/summary/refined/gffcmp/{sp}-cmp.stats"

        #if no gffcompare stats files exist, exit
        found_files=glob.glob(statsfile) #supress error output

        if not found_files:
            raise FileNotFoundError(f"There is no gffcompare statistics files for this species")

        #return data for a concrete stats file
        parsed_dict=gffstats_2data(statsfile)
        #joind data from all stats files
        statsData.append(parsed_dict)
    
    except Exception as e:
                print(f"{sp} presents error: {e}")
                continue

#transform data list to dataframe
df=pd.DataFrame(statsData)
df=df.fillna("NA")

df.to_csv("../results/summary/refined/sum_stats.csv", index=False, sep="\t")
print("Done")


## BUSCO assessment

In [ ]:
#agat for gff>prot
tr_sp=!ls $merged_data
pred_items=os.listdir("../results/refined_pred")
pred_items=[tri.replace("_own.gff3", "_own.gff") for tri in pred_items] #merged data is in _own.gff formats
tr_sp=[species.replace("_own.gff", "") for species in tr_sp if species in pred_items]
print(len(tr_sp))

for src in ["own", "git"]:
    #checker(if things in file)
    fileWritten=False
    filepath=f"../job_commands/refined_buscoScoring_{src}.txt"
    with open(filepath, "w") as f:
        print('cpus="${SLURM_CPUS_PER_TASK:-$(nproc)}"', file=f)
        print(f"mkdir -p ../results/summary/refined/busco", file=f)
        for sp in tr_sp:
            try:
                if src=="git":
                    #if no github parameters exist, dont process this sspecies_source busco evaluation
                    filename_git=f"../training_data/geneid-parameter-files/parameter_files/{sp}*.param"
                    found_files=glob.glob(filename_git) #supress error output

                    if not found_files:
                        raise FileNotFoundError(f"There is no github parameters for this species")
                
                fileWritten=True #if any species has git parameters the job command files will be written
                
                #reference sequence
                RefSeq=!realpath ../training_data/species/$sp/CLEAN_*.fna
                RefSeq=RefSeq[0]

                #annotation prediction by source (own or git)
                pred=!realpath ../results/refined_pred/$sp*_$src*.gff3
                pred=pred[0]

                #create folder to store outputs
                prot_res=f"../results/specie_logs/{sp}/merged/{sp}_prot.fa" #protein sequence output
                busco_outPath=f"../results/refined_busco/" #busco comparisons output
                busco_res=f"{sp}_busco" #busco out name

                #get species taxon
                taxID=sp[sp.rfind("_")+1:]
                #find lineage
                busco_lineage=bob.taxon2lineage(taxID, query_email, f"{busco_database}/file_versions.tsv", "odb12")
                #resolve the NCBI nuclear genetic code for this taxon (table 1 if unresolved)
                gcode=ggc.taxon2geneticcode(taxID, query_email) or 1

                #gff to protein command
                #per-task AGAT config so parallel array tasks don't collide on agat_config.yaml
                agat_cfg=f"agat_{sp}_{src}.yaml"
                expose_cmd=f"agat config --expose --output {agat_cfg} >/dev/null 2>&1"
                print(expose_cmd); print(expose_cmd, file=f)
                TOprotein_cmd=f"agat_sp_extract_sequences.pl -g {pred} -f {RefSeq} -t cds -p --table {gcode} --config {agat_cfg} -o {prot_res}"

                print(TOprotein_cmd)
                print(TOprotein_cmd, file=f)
                clean_cmd=f"rm -f {agat_cfg}"
                print(clean_cmd); print(clean_cmd, file=f)
                #deduplicate protein IDs (AGAT can emit duplicate names that break BUSCO)
                ND_prot_res=prot_res.replace(".fa", "_ND.fa")
                rename_cmd=f"seqkit rename -n {prot_res} > {ND_prot_res}"
                print(rename_cmd)
                print(rename_cmd, file=f)
                
                busco_cmd=f"busco -m protein \
                            -i {ND_prot_res} \
                            --download_path {busco_database} \
                            -l {busco_lineage} \
                            --cpu $cpus \
                            -f \
                            --opt-out-run-stats \
                            --out_path {busco_outPath} \
                            -o {busco_res}" #--offline #autolineage creates errors
                busco_cmd=busco_cmd.replace("                             ", " ")

                busco_plot=f"busco --opt-out-run-stats --plot {busco_outPath}{busco_res}"
                
                print(busco_cmd)
                print(busco_cmd, file=f)
                
                print(busco_plot)
                print(busco_plot, file=f)

                print(f"ln -vf {busco_outPath}{busco_res}/*json ../results/summary/refined/busco/", file=f)

            except Exception as e:
                print(f"{sp} presents error: {e}")
                continue
            
        #image summary
        print(f"busco --opt-out-run-stats --plot ../results/summary/refined/busco/", file=f)
            
    if not fileWritten: #if the file is not written, delete it
        os.remove(filepath)
            

# Summary

The **refined** evaluation outputs are collected under `results/summary/refined/`: the busco and gffcompare cells above write the BUSCO summaries, the gffcompare stats folder and the summary table directly there. The cell below adds the gene/transcript model counts.

In [ ]:
# Gene/transcript model counts for the refined predictions -> results/summary/refined/counts/ (per-species *_gc.txt / *_tc.txt) + counts_summary.tsv.
from scripts.counting_machine import write_counts, CATEGORIES

results_dir = "../results"
summary_dir = f"{results_dir}/summary"
os.makedirs(summary_dir, exist_ok=True)

write_counts(results_dir, summary_dir, "refined", CATEGORIES["refined"])